# Chapter 4 (part 2) — Factor (Categorical) Variables in Regression

Covers: Dummy Variable Representation → Factor Variables with Many Levels → Ordered Factor Variables, on Ames Housing.

The thread: **a regression only understands numbers, so every category gets turned into numbers by *some* coding, and the coding decides what the coefficients mean.** Dummies turn "which group" into "how far from a baseline group". Many-level factors leave rare levels with almost no data behind their coefficients. Ordered factors tempt you into integers, which quietly assume equal gaps between levels.

Data note: quality ratings here are *words* (`Overall_Qual` is `Very_Poor` … `Very_Excellent`, not 1-10), so you'll write the integer mapping yourself.

**How to work each section** (~10 min each; ~1 hour total with the exam): write your one-line prediction in the *Prediction* cell **before** running anything, do the code, then write 2-3 bullets on where the result matched or broke your prediction and why. Ask for a hint if you get stuck.

In [62]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import plotly.express as px

pd.set_option("display.max_columns", None)

ames = pd.read_csv('../data/ames_housing.csv')
ames.shape

(2930, 81)

## 1. Dummy Variables and the Reference Level

To feed a category into a regression you make one 0/1 column per level and leave one level out as the baseline. Numeric picture: average prices are $200k for single-family, $150k for duplexes, $140k for townhouses. With single-family as the baseline, the intercept is $200k, the duplex coefficient is -$50k, the townhouse coefficient is -$60k, and a duplex's prediction is 200 + (-50) = $150k, exactly its group mean. Switch the baseline to townhouse: the intercept becomes $140k, single-family +$60k, duplex +$10k, and the three predictions are still 200, 150, 140. The baseline is a choice about which comparison is easiest to read, not a modeling decision.

**Predict first:** with only `Bldg_Type` in the model, is R² closer to 0.03, 0.15, or 0.40? When you change the baseline, which of these change: the coefficients, R², the fitted values?

**Question 1.** (a) Fit `Sale_Price ~ C(Bldg_Type)`. Name the baseline level and check against the group means that intercept = baseline mean and each coefficient = group mean − baseline mean. (b) Some levels have much bigger standard errors than others. Why? (c) Re-fit with `TwnhsE` as the baseline and confirm which numbers changed and which stayed identical.

*Tools for this question:*
- `C(col)` inside a formula treats `col` as categorical (one dummy per level except the first alphabetically). Coefficient names look like `C(Bldg_Type)[T.Duplex]`.
- `C(col, Treatment(reference="TwnhsE"))` sets the baseline level.
- `df.groupby("col")["y"].mean()` for group means; `df["col"].value_counts()` for group sizes.

_Prediction (one line, before running anything):_

* With only building type in the model, I would imagine the r-squared to be close to 0.40. 
* When you change the baseline, the coefficients are the ones to change because each baseline is a different comparison each time. I would imagine that r-squared and fitted values stay the same

In [46]:
ames["Bldg_Type"].value_counts()

Bldg_Type
OneFam      2425
TwnhsE       233
Duplex       109
Twnhs        101
TwoFmCon      62
Name: count, dtype: int64

In [47]:
# model with dummy variable and no established baseline
model_with_dummy = smf.ols("Sale_Price ~ C(Bldg_Type)", data = ames).fit()
RMSE = np.sqrt((((model_with_dummy.resid)**2).sum())/len(ames))
print(f"Root mean squared error:{RMSE}")
model_with_dummy.summary()

Root mean squared error:78482.02773102747


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             Sale_Price   R-squared:                       0.035
Model:                            OLS   Adj. R-squared:                  0.033
Method:                 Least Squares   F-statistic:                     26.15
Date:                Sat, 17 Oct 2026   Prob (F-statistic):           2.48e-21
Time:                        17:25:04   Log-Likelihood:                -37180.
No. Observations:                2930   AIC:                         7.437e+04
Df Residuals:                    2925   BIC:                         7.440e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
============================================================================================
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                 1.398e+05   7523.637     18.583      0.000    1.25e+05    1.55e+05
C(Bldg_Type)[T.OneFam]      4.5e+04   7690.867      5.851      0.000    2.99e+04    6.01e+04
C(Bldg_Type)[T.Twnhs]    -3874.8764   1.08e+04     -0.357      0.721   -2.51e+04    1.74e+04
C(Bldg_Type)[T.TwnhsE]     5.25e+04   9115.131      5.760      0.000    3.46e+04    7.04e+04
C(Bldg_Type)[T.TwoFmCon] -1.423e+04   1.25e+04     -1.139      0.255   -3.87e+04    1.03e+04
==============================================================================
Omnibus:                     1075.166   Durbin-Watson:                   0.760
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             4852.710
Skew:                           1.728   Prob(JB):                         0.00
Kurtosis:                       8.273   Cond. No.                         16.5
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [48]:
# model with dummy variable and baseline set to twnhse
base_model_with_dummy = smf.ols("Sale_Price ~ C(Bldg_Type, Treatment(reference='TwnhsE'))", data = ames).fit()
RMSE = np.sqrt((((base_model_with_dummy.resid)**2).sum())/len(ames))
print(f"Root mean squared error:{RMSE}")
base_model_with_dummy.summary()

Root mean squared error:78482.02773102747


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             Sale_Price   R-squared:                       0.035
Model:                            OLS   Adj. R-squared:                  0.033
Method:                 Least Squares   F-statistic:                     26.15
Date:                Sat, 17 Oct 2026   Prob (F-statistic):           2.48e-21
Time:                        17:25:04   Log-Likelihood:                -37180.
No. Observations:                2930   AIC:                         7.437e+04
Df Residuals:                    2925   BIC:                         7.440e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
===========================================================================================================================
                                                              coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------
Intercept                                                1.923e+05   5145.921     37.372      0.000    1.82e+05    2.02e+05
C(Bldg_Type, Treatment(reference='TwnhsE'))[T.Duplex]    -5.25e+04   9115.131     -5.760      0.000   -7.04e+04   -3.46e+04
C(Bldg_Type, Treatment(reference='TwnhsE'))[T.OneFam]   -7499.8729   5387.469     -1.392      0.164   -1.81e+04    3063.743
C(Bldg_Type, Treatment(reference='TwnhsE'))[T.Twnhs]    -5.638e+04   9357.841     -6.025      0.000   -7.47e+04    -3.8e+04
C(Bldg_Type, Treatment(reference='TwnhsE'))[T.TwoFmCon] -6.673e+04   1.12e+04     -5.945      0.000   -8.87e+04   -4.47e+04
==============================================================================
Omnibus:                     1075.166   Durbin-Watson:                   0.760
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             4852.710
Skew:                           1.728   Prob(JB):                         0.00
Kurtosis:                       8.273   Cond. No.                         12.5
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

**No Baseline Interpretation**
* In the no baseline model, the model seems to have Duplex building type as the baseline as thats the only building type with no categorical variable assigned meaning that it is represent via the intercept. 
* The variables with the larger standard errors tends to have lower value counts and that is my assumption as to why the standard errors are different. We can se that onefam houses have smaller SE than the other types and it also is the one type with the largest amount of observations. I would think that having more data to that type allows the model to better generalize/capture information related to that building type.

**Baseline Interpretation**
* For the baseline models, coefficients and std error changed. R-squared remained the same.

_What matched, what surprised you, and why (2-3 bullets):_
* It seems like what I predicted was fairly correct. I had predicted that R-squared would remain the same along with the residuals.
* Not to mention, the coefficients did in fact change. My thinking was that the model's coefficients almost acts as a comparison from the baseline. So if the baseline was one fam type, then the coefficients of the other categorical variables would represent the average difference of sales price via building type. For example the model with TwnhsE has the baseline was one where I interpreted as "Homes of building type Duplex on average have a 52500 lower sales price than TwnhsE homes."
* My prediction with R-squared was incorrect since sales price can still widely vary within building types.

## 2. Dummies Alongside a Numeric Predictor

Once a numeric predictor is in the model, a dummy coefficient no longer compares *group averages*; it compares groups **at the same value of the numeric predictor**. Numeric picture: duplexes average $150k at 2,000 sqft; single-family homes average $200k at 1,500 sqft. Raw gap = -$50k. But the duplexes are bigger; if each sqft is worth $100, a 2,000 sqft house "should" be worth $50k more, so *at equal size* the duplex gap is not -$50k but -$100k. Same coefficient label, different question.

**Predict first:** are townhouses (`Twnhs`, `TwnhsE`) smaller or larger than single-family homes on average? Given that, will their coefficients move up or down once `Gr_Liv_Area` is in the model?

**Question 2.** (a) Fit `Sale_Price ~ Gr_Liv_Area + C(Bldg_Type)` and compare each `Bldg_Type` coefficient to Question 1. Which moved most, and in which direction? Explain it with the mean `Gr_Liv_Area` by `Bldg_Type`. (b) In one sentence: what question does the `Duplex` coefficient answer now that it didn't before?

*Tools for this question:*
- `df.groupby("Bldg_Type")["Gr_Liv_Area"].mean()`.

_Prediction (one line, before running anything):_

* I would imagine that town houses are smaller than family homes. I think that once Gr_Liv_Area is added into the model, we will have 2 factors to consider the type of home and the amount of living space and the coefficient of the categorical variables will go down.

In [49]:
model_with_dummy.params

Intercept                   139808.935780
C(Bldg_Type)[T.OneFam]       45003.105457
C(Bldg_Type)[T.Twnhs]        -3874.876374
C(Bldg_Type)[T.TwnhsE]       52502.978383
C(Bldg_Type)[T.TwoFmCon]    -14227.226102
dtype: float64

In [50]:
dummy_numerical_model = base_model_with_dummy = smf.ols("Sale_Price ~ Gr_Liv_Area + C(Bldg_Type)", data = ames).fit()
dummy_numerical_model.params

Intercept                  -48810.937978
C(Bldg_Type)[T.OneFam]      60745.907452
C(Bldg_Type)[T.Twnhs]       42687.010812
C(Bldg_Type)[T.TwnhsE]      88222.492239
C(Bldg_Type)[T.TwoFmCon]     1446.206214
Gr_Liv_Area                   113.898997
dtype: float64

In [51]:
# mean gr_liv_area by building type
ames.groupby("Bldg_Type")["Gr_Liv_Area"].mean().reset_index()

,Bldg_Type,Gr_Liv_Area
0,Duplex,1656.027523
1,OneFam,1517.810309
2,Twnhs,1247.227723
3,TwnhsE,1342.420601
4,TwoFmCon,1518.419355


**Dummy with Numerical and Categorical Features Interpretation**
* Compared to the first model, all the building-type coefficients went **up** and are now all positive. The biggest mover was `Twnhs` (-3.9k to +42.7k): it is about 409 sq ft smaller than a Duplex, and 409 x $113.9 per sq ft is about $46.6k, exactly its shift. Duplex is the baseline and has the most living area (1,656 sq ft), so at equal size every other type looks better than it did on raw averages.
* The negative intercept (-$48.8k) is not a "Duplex coefficient". It is the predicted price of a Duplex with **0 sq ft** of living space, an extrapolation (the smallest home is 334 sq ft), so it says nothing about how duplexes are valued. The model only makes sense inside the range of the data.
* The new question the coefficients answer is: "at the **same living area**, how much more (or less) does this building type sell for than a Duplex?" For example, a OneFam home sells for about $60.7k more than a Duplex of the same size.

_What matched, what surprised you, and why (2-3 bullets):_

* My prediction was half right. Townhouses are in fact smaller (1,247 and 1,342 sq ft vs. 1,656 for Duplex), but the coefficients went **up**, not down. I thought adding living area would pull the building-type coefficients down; instead, because Duplex is the baseline and the biggest type, controlling for size lifted every other type relative to it.
* Townhouses now have the highest coefficients, but that does not show they are worth more per sq ft. This model forces **one slope** ($113.9 per sq ft) on every building type, so it cannot say that. Testing whether the value per sq ft differs by type needs an interaction term.

## 3. Factor Variables with Many Levels

`Neighborhood` has 28 levels, from North Ames (443 homes) down to Landmark (1 home). A dummy for a one-house neighborhood fits that house *perfectly*: its coefficient is exactly that house's price gap, with nothing to tell signal from noise. The book's remedy is to consolidate levels: fit a model without the factor, look at the residuals grouped by level, and merge levels with similar residuals into a few bins.

Leakage warning: if you compute those bins on all the data and then evaluate on the same data, the grouping has already "seen" the answers. The honest version builds the bins on the *training* split only and applies them to the test split.

**Predict first:** the 28 dummies will beat "no neighborhood" on test RMSE. How much of that gain will the 5 tiers keep: almost none, about half, or nearly all?

**Question 3.** (a) Fit `Sale_Price ~ C(Neighborhood)` and look at the coefficients and standard errors of the five smallest neighborhoods. What do you see? (b) Split train/test 70/30 with a fixed seed. On the **train set only**, fit `Sale_Price ~ Gr_Liv_Area + Year_Built`, take the median residual per neighborhood, and cut the neighborhoods into 5 tiers by those medians. (c) Fit three models on train and compare **test RMSE**: no neighborhood information, 28 neighborhood dummies, the 5-tier factor (all with `Gr_Liv_Area + Year_Built`). Which wins, and why?

*Tools for this question:*
- `df["Neighborhood"].value_counts()`; `df.assign(resid=res.resid).groupby("Neighborhood")["resid"].median()`.
- `pd.qcut(series, q=5, labels=False)` cuts into 5 equal-count bins; `df["col"].map(dict_or_series)` attaches the tiers.
- Gotcha: a neighborhood that's in test but not in train makes `res.predict(test)` raise a confusing `PatsyError`, and maps to `NaN` in the tier column. Drop those rows first: `test[test["Neighborhood"].isin(train["Neighborhood"].unique())]`.

_Prediction (one line, before running anything):_

* I predict that 28 neighborhoods will beat no neighborhood on RMSE. I imagine the 5 tiers may keep half of the gain.

In [52]:
# neighborhood value counts
ames["Neighborhood"].value_counts()[-5:]

Neighborhood
Northpark_Villa    23
Blueste            10
Greens              8
Green_Hills         2
Landmark            1
Name: count, dtype: int64

In [53]:
level_factors_model = smf.ols("Sale_Price ~ C(Neighborhood)", data = ames).fit()
level_factors_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             Sale_Price   R-squared:                       0.573
Model:                            OLS   Adj. R-squared:                  0.569
Method:                 Least Squares   F-statistic:                     144.4
Date:                Sat, 17 Oct 2026   Prob (F-statistic):               0.00
Time:                        17:25:04   Log-Likelihood:                -35984.
No. Observations:                2930   AIC:                         7.202e+04
Df Residuals:                    2902   BIC:                         7.219e+04
Df Model:                          27                                         
Covariance Type:            nonrobust                                         
==============================================================================================================================
                                                                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------------------------
Intercept                                                   1.967e+05   9907.848     19.849      0.000    1.77e+05    2.16e+05
C(Neighborhood)[T.Blueste]                                 -5.307e+04   1.93e+04     -2.748      0.006   -9.09e+04   -1.52e+04
C(Neighborhood)[T.Briardale]                               -9.105e+04   1.38e+04     -6.609      0.000   -1.18e+05    -6.4e+04
C(Neighborhood)[T.Brookside]                               -7.191e+04   1.11e+04     -6.467      0.000   -9.37e+04   -5.01e+04
C(Neighborhood)[T.Clear_Creek]                                1.2e+04   1.27e+04      0.947      0.344   -1.29e+04    3.69e+04
C(Neighborhood)[T.College_Creek]                            5141.7559   1.04e+04      0.494      0.622   -1.53e+04    2.56e+04
C(Neighborhood)[T.Crawford]                                 1.089e+04   1.12e+04      0.975      0.330    -1.1e+04    3.28e+04
C(Neighborhood)[T.Edwards]                                 -6.582e+04   1.06e+04     -6.210      0.000   -8.66e+04    -4.5e+04
C(Neighborhood)[T.Gilbert]                                 -6015.1028   1.07e+04     -0.561      0.575    -2.7e+04     1.5e+04
C(Neighborhood)[T.Green_Hills]                              8.334e+04   3.84e+04      2.172      0.030    8097.376    1.59e+05
C(Neighborhood)[T.Greens]                                  -3130.4286    2.1e+04     -0.149      0.882   -4.43e+04    3.81e+04
C(Neighborhood)[T.Iowa_DOT_and_Rail_Road]                  -9.291e+04   1.13e+04     -8.221      0.000   -1.15e+05   -7.07e+04
C(Neighborhood)[T.Landmark]                                -5.966e+04   5.34e+04     -1.118      0.264   -1.64e+05     4.5e+04
C(Neighborhood)[T.Meadow_Village]                          -1.009e+05   1.31e+04     -7.684      0.000   -1.27e+05   -7.52e+04
C(Neighborhood)[T.Mitchell]                                -3.444e+04   1.11e+04     -3.114      0.002   -5.61e+04   -1.28e+04
C(Neighborhood)[T.North_Ames]                              -5.156e+04   1.02e+04     -5.047      0.000   -7.16e+04   -3.15e+04
C(Neighborhood)[T.Northpark_Villa]                         -5.595e+04   1.48e+04     -3.792      0.000   -8.49e+04    -2.7e+04
C(Neighborhood)[T.Northridge]                               1.337e+05   1.17e+04     11.424      0.000    1.11e+05    1.57e+05
C(Neighborhood)[T.Northridge_Heights]                       1.254e+05   1.07e+04     11.704      0.000    1.04e+05    1.46e+05
C(Neighborhood)[T.Northwest_Ames]                          -8254.7702   1.09e+04     -0.756      0.450   -2.97e+04    1.31e+04
C(Neighborhood)[T.Old_Town]                                -7.267e+04   1.05e+04     -6.939      0.000   -9.32e+04   -5.21e+04
C(Neighborhood)[T.Sawyer]          

* The neighborhoods with the fewest homes (`Landmark` 1, `Green_Hills` 2, `Greens` 8, `Blueste` 10, `Northpark_Villa` 23) tend to have the largest standard errors, because the model has so little information to fit them.
* Their coefficients are also basically the gap of 1-2 houses: `Green_Hills` is +$83k and `Landmark` is -$60k. That is noise, not a reliable neighborhood effect.

In [54]:
train_df, test_df = train_test_split(
    ames, test_size=0.2, random_state=42
)

# fit the first model
no_neighborhood_model = smf.ols("Sale_Price ~ Gr_Liv_Area + Year_Built", data = train_df).fit()

# get the residual and tag it back to the row it belonged to
train_df = train_df.assign(resid = no_neighborhood_model.resid)

# sort neighborhoods by median residual
neighborhood_medians = (
    train_df.groupby("Neighborhood")["resid"]
    .median()
    .to_frame(name="med_resid")
)

# look up table with the medians and tiers
neighborhood_medians["Tier"] = pd.qcut(
    neighborhood_medians["med_resid"], q=5, labels=[1, 2, 3, 4, 5]
)

train_df = train_df.join(neighborhood_medians["Tier"], on="Neighborhood")
test_df = test_df.join(neighborhood_medians["Tier"], on="Neighborhood")

train_df

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,Land_Slope,Neighborhood,Condition_1,Condition_2,Bldg_Type,House_Style,Overall_Qual,Overall_Cond,Year_Built,Year_Remod_Add,Roof_Style,Roof_Matl,Exterior_1st,Exterior_2nd,Mas_Vnr_Type,Mas_Vnr_Area,Exter_Qual,Exter_Cond,Foundation,Bsmt_Qual,Bsmt_Cond,Bsmt_Exposure,BsmtFin_Type_1,BsmtFin_SF_1,BsmtFin_Type_2,BsmtFin_SF_2,Bsmt_Unf_SF,Total_Bsmt_SF,Heating,Heating_QC,Central_Air,Electrical,First_Flr_SF,Second_Flr_SF,Low_Qual_Fin_SF,Gr_Liv_Area,Bsmt_Full_Bath,Bsmt_Half_Bath,Full_Bath,Half_Bath,Bedroom_AbvGr,Kitchen_AbvGr,Kitchen_Qual,TotRms_AbvGrd,Functional,Fireplaces,Fireplace_Qu,Garage_Type,Garage_Finish,Garage_Cars,Garage_Area,Garage_Qual,Garage_Cond,Paved_Drive,Wood_Deck_SF,Open_Porch_SF,Enclosed_Porch,Three_season_porch,Screen_Porch,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Year_Sold,Sale_Type,Sale_Condition,Sale_Price,Longitude,Latitude,resid,Tier
381,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,80,10400,Pave,No_Alley_Access,Regular,Lvl,AllPub,Inside,Gtl,Northwest_Ames,Norm,Norm,OneFam,One_Story,Good,Average,1976,1976,Gable,CompShg,HdBoard,HdBoard,BrkFace,189,Typical,Typical,CBlock,Good,Typical,No,Unf,7,Unf,0,1090,1090,GasA,Typical,Y,SBrkr,1370,0,0,1370,0,0,2,0,3,1,Typical,6,Typ,1,Typical,Attchd,RFn,2,479,Typical,Typical,Paved,0,0,0,0,0,0,No_Pool,Minimum_Privacy,NaN,0,6,2009,WD,Family,152000,-93.634694,42.049654,-20947.498811,2
834,Two_Story_1946_and_Newer,Residential_Low_Density,0,28698,Pave,No_Alley_Access,Moderately_Irregular,Low,AllPub,CulDSac,Sev,Clear_Creek,Norm,Norm,OneFam,Two_Story,Average,Average,1967,1967,Flat,Tar&Grv,Plywood,Plywood,NaN,0,Typical,Typical,PConc,Typical,Good,Gd,LwQ,4,ALQ,764,0,1013,GasA,Typical,Y,SBrkr,1160,966,0,2126,0,1,2,1,3,1,Typical,7,Min2,0,No_Fireplace,Attchd,Fin,2,538,Typical,Typical,Paved,486,0,0,0,225,0,No_Pool,No_Fence,NaN,0,6,2009,WD,Abnorml,185000,-93.682565,42.025546,-49064.349566,4
1898,Duplex_All_Styles_and_Ages,Residential_Low_Density,70,9842,Pave,No_Alley_Access,Regular,Lvl,AllPub,FR2,Gtl,North_Ames,Norm,Norm,Duplex,One_Story,Below_Average,Average,1962,1962,Gable,CompShg,HdBoard,HdBoard,NaN,0,Typical,Typical,Slab,No_Basement,No_Basement,No_Basement,No_Basement,5,No_Basement,0,0,0,GasA,Typical,Y,SBrkr,1224,0,0,1224,0,0,2,0,2,2,Typical,6,Typ,0,No_Fireplace,CarPort,Unf,2,462,Typical,Typical,Paved,0,0,0,0,0,0,No_Pool,No_Fence,NaN,0,3,2007,WD,Normal,101800,-93.621368,42.041332,-42563.863016,3
678,Duplex_All_Styles_and_Ages,Residential_Low_Density,60,7200,Pave,No_Alley_Access,Regular,Lvl,AllPub,Inside,Gtl,North_Ames,Norm,Norm,Duplex,One_Story,Below_Average,Average,1949,1950,Gable,CompShg,BrkFace,Stone,NaN,0,Typical,Typical,Slab,No_Basement,No_Basement,No_Basement,No_Basement,5,No_Basement,0,0,0,Wall,Fair,N,FuseF,1040,0,0,1040,0,0,2,0,2,2,Typical,6,Typ,0,No_Fireplace,Detchd,Unf,2,420,Typical,Typical,Paved,0,0,0,0,0,0,No_Pool,No_Fence,NaN,0,6,2009,WD,Normal,90000,-93.608903,42.035841,-23292.111606,3
700,Two_Family_conversion_All_Styles_and_Ages,Residential_Medium_Density,63,7627,Pave,No_Alley_Access,Regular,Lvl,AllPub,Corner,Gtl,Old_Town,Artery,Norm,TwoFmCon,Two_Story,Below_Average,Above_Average,1920,1950,Gable,CompShg,AsbShng,AsbShng,NaN,0,Fair,Typical,BrkTil,Fair,Poor,No,Unf,7,Unf,0,600,600,GasA,Good,N,SBrkr,1101,600,0,1701,0,0,2,0,4,2,Fair,8,Typ,0,No_Fireplace,No_Garage,No_Garage,0,0,No_Garage,No_Garage,Dirt_Gravel,0,0,148,0,0,0,No_Pool,No_Fence,NaN,0,10,2009,WD,Normal,94550,-93.620068,42.030474,-49648.188355,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1638,Two_Family_conversion_All_Styles_and_Ages,Residential_Low_Density,195,41600,Pave,No_Alley_Access,Slightly_Irregular,Lvl

In [ ]:
from sklearn.metrics import mean_squared_error
# compute RMSE for each of the 3 models

# no neighborhood
test_predictions = no_neighborhood_model.predict(test_df)
y_true_test = test_df["Sale_Price"]
test_rmse = np.sqrt(mean_squared_error(y_true_test, test_predictions))
print(f"Test RMSE no neighborhood: {test_rmse:.4f}")


# 27 dummies
neighborhood_model = smf.ols("Sale_Price ~ Gr_Liv_Area + Year_Built + C(Neighborhood)", data = train_df).fit()
test_predictions = neighborhood_model.predict(test_df)
test_rmse = np.sqrt(mean_squared_error(y_true_test, test_predictions))
print(f"Test RMSE neighborhood (27 dummies): {test_rmse:.4f}")

# median tiers
tier_model = smf.ols("Sale_Price ~ Gr_Liv_Area + Year_Built + C(Tier)", data = train_df).fit()
test_predictions = tier_model.predict(test_df)
test_rmse = np.sqrt(mean_squared_error(y_true_test, test_predictions))
print(f"Test RMSE neighborhood (5 median tiers): {test_rmse:.4f}")


Test RMSE no neighborhood: 52064.7379
Test RMSE neighborhood (27 dummies): 42624.3464
Test RMSE neighborhood (4 median tiers): 43640.0400


In [56]:
print(neighborhood_model.rsquared)
print(tier_model.rsquared)

0.7571946093683747
0.7374428428223907


* The 27-dummy model won on test (RMSE 42,624 vs. 43,640 for the 5 tiers and 52,065 for no neighborhood), but only by about $1,000 (2.4%). The tiers keep about **89%** of the dummies' gain using 4 parameters instead of 27.
* In-sample R-squared (0.757 vs. 0.737) can't be compared directly, since more variables always raise it. Test RMSE is the fair comparison.
* The dummies probably didn't overfit badly because no test neighborhood was missing from train (nothing new to extrapolate to), and the 5 smallest neighborhoods hold only 44 of 2,930 homes.
* The median residual is not "which neighborhoods have the most errors". It is the signed gap between what homes there sell for and what size and age predict, so it ranks neighborhoods by how pricey they are. That is exactly the information the model needs, which is why 5 tiers capture most of it. A weakness: the most extreme medians (`Green_Hills`, `Landmark`) come from 1-2 homes, so where tiny neighborhoods land in the tiers is itself noisy.

_What matched, what surprised you, and why (2-3 bullets):_

* My prediction that neighborhood information beats none was correct (test RMSE 52,065 down to 42,624 with dummies). But I predicted the 5 tiers would keep about half of the gain, and they kept about **89%** (43,640).
* I was surprised that the model with 27 dummies performed about the same as the tiered one. Most of the neighborhood signal is one idea (how pricey the area is for its size and age), and 5 tiers capture it with 4 parameters.

## 4. Ordered Factor Variables

`Overall_Qual` runs from `Very_Poor` to `Very_Excellent`, ten ordered words. Two natural encodings: integers 1-10 (one slope: "each step up in quality is worth $X") or dummies (nine separate steps). Numeric picture: if the integer model says each step is worth $30k, it predicts the same $30k gap from Poor to Fair as from Excellent to Very_Excellent. The dummy version would let those gaps be $5k and $100k if that's what the data says, at the cost of eight extra parameters.

**Predict first:** as quality goes up, will the dollar gap from one level to the next grow, shrink, or stay roughly constant?

**Question 4.** (a) Map the ten level names to 1-10 and fit `Sale_Price ~ qual_int` and `Sale_Price ~ C(Overall_Qual)`. Plot the mean price per quality level against the integer model's straight line. Where does the line fail? (b) Compute the 9 step sizes (mean price at level k+1 minus level k). Are they equal? (c) Which model would you use for a one-sentence stakeholder summary, and which for the best predictions?

*Tools for this question:*
- `series.map({"Very_Poor": 1, "Poor": 2, ...})` for the integer coding (check exact spellings with `value_counts()`); Group by your integer column, not the word column: `df.groupby("qual_int")["Sale_Price"].mean().diff()`. Grouping by `Overall_Qual` sorts the words alphabetically (`Above_Average`, `Average`, `Below_Average`, ...), which scrambles both the step sizes and the plot.

_Prediction (one line, before running anything):_

In [57]:
ames["Overall_Qual"].value_counts()

Overall_Qual
Average           825
Above_Average     732
Good              602
Very_Good         350
Below_Average     226
Excellent         107
Fair               40
Very_Excellent     31
Poor               13
Very_Poor           4
Name: count, dtype: int64

* My prediction is that the middle quality tiers like above,below, and at average will stay reasonably within a similar range and then go up considerably when it rises to good quality. And I believe the rise from poor to average will also be a considerable jump. 

In [65]:
# build quality map
qual_map = {
    "Very_Poor": 1,
    "Poor": 2,
    "Fair": 3,
    "Below_Average": 4,
    "Average": 5,
    "Above_Average": 6,
    "Good": 7,
    "Very_Good": 8,
    "Excellent": 9,
    "Very_Excellent": 10,
}

# map the qualities
ames["Quality_Tier"] = ames["Overall_Qual"].map(qual_map)

# fit the models
raw_qual_model = smf.ols("Sale_Price ~ Overall_Qual", data = ames).fit()
tier_qual_model = smf.ols("Sale_Price ~ Quality_Tier", data = ames).fit()

# average sales price per tier
avg_tier_prices = pd.DataFrame(ames.groupby("Quality_Tier")["Sale_Price"].mean().reset_index())

# fit the average tier price 
avg_tier_prices["Fitted_Sale_Price"] = tier_qual_model.predict(avg_tier_prices["Quality_Tier"])

# plot
fig = px.line(x = avg_tier_prices["Quality_Tier"], y = avg_tier_prices["Fitted_Sale_Price"], title = "Fitted sales price vs tier" )
fig.update_traces(name="Fitted Values Line", showlegend=True)

fig.add_scatter(
    x=avg_tier_prices["Quality_Tier"],
    y=avg_tier_prices["Sale_Price"],
    mode="markers", 
    name="Avg Actual Sale Price",
    marker=dict(size=12, color="crimson", symbol="diamond"),  # Make markers pop
)

fig.show()


In [76]:
fig = px.line(
    avg_tier_prices,
    x="Quality_Tier",
    y="Sale_Price",
    title="Tier vs. Sale Price (Average)",
    labels={
        "Quality_Tier": "Quality Tier",
        "Sale_Price": "Average Sale Price",
    },
    markers=True
)

fig.update_traces(
    textposition="top center", 
    mode="lines+markers+text"   # Enables lines, markers, and text simultaneously
)

fig.show()

In [77]:
# model results
raw_qual_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             Sale_Price   R-squared:                       0.702
Model:                            OLS   Adj. R-squared:                  0.701
Method:                 Least Squares   F-statistic:                     765.2
Date:                Sat, 17 Oct 2026   Prob (F-statistic):               0.00
Time:                        20:36:32   Log-Likelihood:                -35457.
No. Observations:                2930   AIC:                         7.093e+04
Df Residuals:                    2920   BIC:                         7.099e+04
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
==================================================================================================
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                       1.621e+05   1613.656    100.474      0.000    1.59e+05    1.65e+05
Overall_Qual[T.Average]        -2.738e+04   2216.809    -12.350      0.000   -3.17e+04    -2.3e+04
Overall_Qual[T.Below_Average]  -5.565e+04   3322.305    -16.749      0.000   -6.22e+04   -4.91e+04
Overall_Qual[T.Excellent]       2.062e+05   4518.560     45.635      0.000    1.97e+05    2.15e+05
Overall_Qual[T.Fair]           -7.894e+04   7089.075    -11.136      0.000   -9.28e+04    -6.5e+04
Overall_Qual[T.Good]             4.29e+04   2402.097     17.857      0.000    3.82e+04    4.76e+04
Overall_Qual[T.Poor]           -1.098e+05   1.22e+04     -8.989      0.000   -1.34e+05   -8.59e+04
Overall_Qual[T.Very_Excellent]  2.881e+05   8005.572     35.986      0.000    2.72e+05    3.04e+05
Overall_Qual[T.Very_Good]       1.088e+05   2837.204     38.342      0.000    1.03e+05    1.14e+05
Overall_Qual[T.Very_Poor]      -1.134e+05   2.19e+04     -5.181      0.000   -1.56e+05   -7.05e+04
==============================================================================
Omnibus:                      574.592   Durbin-Watson:                   1.468
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             4433.913
Skew:                           0.715   Prob(JB):                         0.00
Kurtosis:                       8.854   Cond. No.                         29.2
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [78]:
tier_qual_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             Sale_Price   R-squared:                       0.639
Model:                            OLS   Adj. R-squared:                  0.639
Method:                 Least Squares   F-statistic:                     5179.
Date:                Sat, 17 Oct 2026   Prob (F-statistic):               0.00
Time:                        20:36:43   Log-Likelihood:                -35740.
No. Observations:                2930   AIC:                         7.148e+04
Df Residuals:                    2928   BIC:                         7.150e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept      -9.5e+04   3933.822    -24.150      0.000   -1.03e+05   -8.73e+04
Quality_Tier  4.525e+04    628.805     71.964      0.000     4.4e+04    4.65e+04
==============================================================================
Omnibus:                      953.364   Durbin-Watson:                   1.398
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             6610.719
Skew:                           1.362   Prob(JB):                         0.00
Kurtosis:                       9.836   Cond. No.                         28.4
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [79]:
# held-out RMSE: one integer slope vs. one dummy per quality level
from sklearn.metrics import mean_squared_error

q_train, q_test = train_test_split(ames, test_size=0.2, random_state=42)

for name, formula in [("integer tier", "Sale_Price ~ Quality_Tier"),
                      ("dummy per level", "Sale_Price ~ C(Overall_Qual)")]:
    res = smf.ols(formula, data=q_train).fit()
    rmse = np.sqrt(mean_squared_error(q_test["Sale_Price"], res.predict(q_test)))
    print(f"Test RMSE, {name}: {rmse:,.0f}")

Test RMSE, integer tier: 52,879
Test RMSE, dummy per level: 46,911


**Ordered Features**
* A: Looking at the plotted average sales price of the tiers along with the fitted values, the line fails towards the lowest and highest tiers. Those also happen to be the tiers with the lowest amount of observations. The main reason is curvature: the line assumes a constant ~$45k per tier, so it predicts about -$50k for `Very_Poor` (actual $48.7k) and is about $93k too low for `Very_Excellent` (line $357.5k, actual $450.2k).
* B: Across the different tiers the step sizes are not equal. They run from about $3.6k (`Very_Poor` to `Poor`) up to $97k (`Very_Good` to `Excellent`): steady at about $23k-$28k through the middle tiers, then growing from `Good` upward. The integer model forces one constant $45k step.
* C: For a one-sentence stakeholder summary I would use the integer tier model: "one tier up is worth about $45k more on average", with the caveat that it hides how uneven the steps are. For the best predictions I would use the dummy model. Its R-squared is higher (0.702 vs. 0.639), and that is not just from having more features: adjusted R-squared (which penalizes features) is still 0.701 vs. 0.639, AIC is about 550 lower, and on held-out data the test RMSE is $46,911 vs. $52,879 for the integer model (about 11% better).
* The standard errors do not settle this. The tier slope's SE is small because one slope is estimated from all 2,930 homes, not because the model predicts better. SE measures how precisely a coefficient is estimated, not how well the model predicts. The big SEs on rare tiers (`Very_Excellent`, about $8k) just reflect how few homes are behind them.

_What matched, what surprised you, and why (2-3 bullets):_
* I originally thought that there would also be a big jump in average sales price from lowest quality to the middle tier. But there seems to be only a gradual climb and then a steep climb from the middle to higher tiers.
* I also expected the R-squared for the tiered model to be better, but it was lower (0.639 vs. 0.702). I first explained that as "1 feature vs. 10", but adjusted R-squared (0.701 vs. 0.639) and held-out RMSE ($46,911 vs. $52,879) show the dummies are genuinely better, because the steps between tiers really are uneven. I had also used SE to pick a model, but SE measures coefficient precision, not prediction quality; held-out RMSE is the right check.

---
# Exam — check your intuition

5 questions, about 3-4 minutes each. Almost none need code: the model outputs are printed for you as text, and you're asked to *read*, *predict*, or *explain*. Scratch code is fine if you want to poke at something, but try to answer from understanding first.

How it works:
- Write your answer in the **Your answer** cell under each question. Plain words, 2-3 bullets — the way you'd say it out loud in an interview.
- Leave the **Review** cell alone. When you're done, ask Claude to review; it fills those cells in. Being right is a perfectly valid review outcome.
- Each question is tagged with the role it leans toward — **DS** (general data science), **Causal** (decision science / causal inference), **ML** (machine learning). Anything tagged for one role is still fair game for the others.
- Several questions are phrased the way interviewers ask them. A good answer is short, names the mechanism, and says what you'd *do* about it.

### Exam Q1 — Reading dummy coefficients
*Role lean: DS*

Here is `Sale_Price ~ C(Foundation)` on all Ames homes. (`statsmodels` silently picked `BrkTil` as the reference level because it comes first alphabetically.)

```text
                             coef  std err     t   P>|t|
Intercept                128107.0   3847.0  33.0  <0.001
C(Foundation)[T.CBlock]   20177.0   4301.0   5.0  <0.001
C(Foundation)[T.PConc]    98962.0   4279.0  23.0  <0.001
C(Foundation)[T.Slab]    -17650.0  10427.0  -2.0   0.091
C(Foundation)[T.Stone]    21680.0  20812.0   1.0   0.298
C(Foundation)[T.Wood]     52793.0  30581.0   2.0   0.084

R-squared = 0.280
Homes per foundation: PConc 1310, CBlock 1244, BrkTil 311, Slab 49, Stone 11, Wood 5
```

(a) What is the average sale price of **Slab** homes, and how do you know from the table alone? (b) Slab's coefficient is -17,650 and Wood's is +52,793, yet *neither* has a small p-value while CBlock's +20,177 does. How can a difference of similar or larger size (Wood) be "less significant"? (c) Without re-fitting: if we change the reference level to `PConc`, what will the new intercept be, what will the `CBlock` coefficient become, and what will R-squared be? (d) Would changing the reference level change any *prediction* the model makes? Why do people still care which level is the reference?

**Your answer (Q1):**

* A: Average sales price of slab homes would be 128107 + (-17650(1)) = 110457. I know from the table alone because the only factor to consider is foundation type and we already know that the table is referencing BrkTil as the base so really the equation is showing us that, on average, slab foundation homes have a sales price that is 17.65k lower than brick tile.
* B: I would imagine that the number of observations plays a big role in this. Wood, although it has a larger coefficient, has only 5 observations with the highest standard error. So the model deems it as less significant.
* C: The new coefficient will essentially represent the average sales price of PConc homes. Cblock will be a coefficient that represents the average sales price difference from the reference which would be PConc. R-squared will remain the same.
* D: I believe RMSE remains the same when reference changes so I would imagine that predictions remain consistent. People care about the reference because it may matter to the case being studied. The regression results allow us to explain the differences in average sale price based on the baseline of interest.

**Review (Q1):**

**Overall: strong. Only (c) needs numbers.**

* **(a) Right.** 128,107 − 17,650 = **$110,457**.
* **(b) Right.** Significance is coefficient ÷ SE. Wood's t is only about 1.7 because 5 homes make its SE ($30.6k) huge.
* **(c) Mostly right.** Add the numbers: new intercept = PConc mean = 128,107 + 98,962 = **$227,069**; `CBlock` becomes 20,177 − 98,962 = **−$78,785**; R² stays 0.280.
* **(d) Right.** Predictions don't change. The reference decides *which comparison* is easy to read, and which p-values you see.

### Exam Q2 — The dummy-variable trap
*Role lean: DS · ML · interview*

Foundation has 6 levels. (a) Suppose you build 6 dummy columns (one per level) and **also** keep an intercept. Why does ordinary least squares break (or give nonsense) — use a concrete row, e.g. a PConc house, to say what the columns add up to. (b) Suppose instead you keep all 6 dummies and **drop the intercept**. What does each coefficient now equal, and what has to be different about how you read R-squared in that model? (c) Interview version: "Why does one-hot encoding for linear regression drop a column, but a random forest doesn't care?" Answer in two or three sentences.

**Your answer (Q2):**

* A: I would say that the OLS breaks becuase there is no reference so the intercept is essentially meaningless/has nothing to build off of from which the other coefficient are meant to be constructed off of the reference.
* B: I would imagine that dropping the intercept but keeping all the 6 dummies essential represents the average sales price of each foundation type. In regards to R-squared, the reading is different now that the denominator is change to essential the sum of squared totals with no addition of the y-hat since its eseentially 0 at this point. 
* C: One hot encoding drops a column for linear regression in order to establish a baseline. Otherwise, there would be perfect multicolinearity and then you cannot separate the effect of foundation types from the intercept. Random forests dont care becuase they dont build coefficients.

**Review (Q2):**

**Overall: (b) and (c) are good; (a) misses the mechanism.**

* **(a) Off.** The problem isn't a missing reference. The 6 dummies sum to 1 on every row, which is exactly the intercept column. A PConc house is (0, 0, 1, 0, 0, 0), which sums to 1, the same as the intercept's 1. *Perfectly collinear columns mean infinitely many coefficient sets fit equally well* (add $10k to the intercept, subtract $10k from every dummy).
* **(b) Mostly right.** Each coefficient is that level's mean (PConc = $227,069). For R², many packages (e.g., R's `lm`) measure distance from 0 instead of from the mean, which inflates it. `statsmodels` spots the implicit constant and still reports 0.28 here, so check what your tool does.
* **(c) Right.**

### Exam Q3 — Factors with many levels
*Role lean: DS · ML*

`Exterior_1st` has 16 levels. This is a slice of `Sale_Price ~ C(Exterior_1st)` showing only the seven rarest levels (reference level is `AsbShng`, 44 homes):

```text
         coef vs AsbShng  std err  homes
ImStucc         160499.0  73799.0    1.0
PreCast         158499.0  73799.0    1.0
CBlock           -8251.0  52761.0    2.0
AsphShn         -10314.0  52761.0    2.0
Stone           156999.0  52761.0    2.0
BrkComm          35999.0  31758.0    6.0
Stucco           48259.0  15648.0   43.0
```

(a) Why are the standard errors so large for the levels with 1-2 homes, and what, concretely, is the model doing to fit a 1-home level perfectly? (b) Name two reasonable ways to handle a factor like this before putting it in a regression (think: what do you do with the rare levels, and what do you do about levels that behave similarly). (c) A colleague proposes replacing the factor with "the mean `Sale_Price` of each exterior type" as one numeric column. Name one big advantage and one serious danger of that approach.

**Your answer (Q3):**

* A: My thinking is that SE is large for low observation categories becuase there's simply not enough examples to be able to generalize and make predictions on newer unseen data. The model is essentially assigning the sale price of the 1 observation it saw to be the coefficient.
* B: A common approach is to combine the exterior into tiers based on something like median sales price and then use the tiers as your variables for the models. So now what was a variable that had 1 observation might be a tier with more observations of exterior that were grouped together based on median sale price.
* C: One big advantage I guess would be that you're effectively doing what the model already does in a sense. But if I had to name a danger I would consider thinking about the baseline. Currently, the model represents something like "the difference in average sales price of a home of this exterior from AsbShng." using the average sale price of each exterior type as the feature itself could change the interpretation of the regression.

**Review (Q3):**

**Overall: (a) is good; (b) and (c) are each half right.**

* **(a) Right.** A 1-home level's coefficient *is* that house's price gap, so it fits perfectly, and the SE is huge ($73.8k vs. $15.6k for 43 homes).
* **(b) Half right.** Merging similar levels into tiers is one way (build the tiers on **training data only**). Missing the other: lump rare levels (say, under 20 homes) into an **"Other"** bucket.
* **(c) Advantage: mostly right; danger: Off.** Advantage: one column instead of 15, with the same group-mean predictions. The serious danger is **leakage**: each house's own price sits inside its group mean, so training fit looks better than it is, worst for tiny groups. *Compute the means from training data only.*

### Exam Q4 — Treating an ordered factor as a number
*Role lean: DS · ML · interview*

`Kitchen_Qual` is ordered (Fair < Typical < Good < Excellent). Someone codes it 1-4 and fits a single slope; you can compare to the model with one dummy per level:

```text
Model: Sale_Price ~ kq   (the single numeric column kq; 1 'Poor' home dropped)
Intercept = -23,456   slope on kq = 81,317   R-squared = 0.454

Kitchen_Qual  coded as  homes  actual mean price  linear-coding fitted
        Fair         1     70             105907                 57861
     Typical         2   1494             139550                139177
        Good         3   1160             210836                220494
   Excellent         4    205             337339                301810

Same data with one dummy per level (C(Kitchen_Qual)): R-squared = 0.482
```

(a) What does the linear coding force to be true about the gaps between adjacent quality levels, and where in the table can you see that it's not true? (b) The dummy model has higher R-squared with 2 extra parameters. Is that an argument for the dummies, against, or neither — and what would you check to decide? (c) Interview version: "For ordinal features, when would you choose integer coding, one-hot, or something else, and does your answer change between linear regression and gradient-boosted trees?" 

**Your answer (Q4):**

* A: Linear coding assumes that the change from the level to the next is consistent or moves in a straight line. However, we have seen this not to be the case where something like quality moves flat, then steadily up, and then jumps to a considerably steeper step up. These varying steps aren't capture well by linear coding the levels. We can see this to be the case becuase of how fast the actual mean price jumps between kitchen qualities.
* B: for this specific case, the model will most likely make better predictions using dummies simply becuase the steps are not equal. Checking SE, adjusted R-squared for differences in feature count, and heldout RMSE could help determine which model is better for predictions.
* C: For ordinal features and linear regression, I would choose based on the context of the column. Columns that genuinely represent tiers can benefit from ordinal encoding while columns that simply represent the existence of a state might benefit from one hot encoding/integer coding. Generally, for Gradient Boosted trees, encoding your categorical variables might not even be needed because they are usually handled natively within the library or the nature of tree based models anyways.

**Review (Q4):**

**Overall: good concepts. Use the table's numbers, and swap SE for AIC in (b).**

* **(a) Right idea, cite the table.** Linear coding forces one gap of **$81,317** per step. The actual gaps are $33.6k, $71.3k and $126.5k, so they keep growing. The fit misses: Fair under by $48k, Excellent under by $36k, Good over by $10k.
* **(b) Mostly right.** R² alone is **neither** for nor against, since it always rises with extra parameters. Adjusted R² and held-out RMSE are right; add AIC or an F-test of the nested models. SE doesn't measure prediction quality (see Section 4).
* **(c) Mostly right.** Linear model: integer coding only if gaps are about equal, otherwise dummies. **Trees: integer coding is fine**, since they only use the order. "Existence of a state" isn't an ordinal feature.

### Exam Q5 — Target encoding and leakage (supplementary)
*Role lean: ML · DS · interview*

A high-cardinality factor like `Neighborhood` (28 levels) is converted into one numeric column: each house gets the **mean Sale_Price of its neighborhood**. Two ways to compute those means, each used as the *prediction itself* for 500 held-out homes (averaged over 30 random splits; the 'train-only' means are computed from just 150 homes):

```text
neighborhood means computed from ALL 2,930 homes (including the held-out ones):   held-out R-squared = 0.585
neighborhood means computed only from the 150 training homes:                       held-out R-squared = 0.470
```
(a) Why is the first number higher, and which one tells you how the feature would do on genuinely new data? (b) Why would the gap be even *bigger* if the neighborhoods were smaller (say, 3 homes each)? (c) Describe in two or three sentences how you'd build this feature correctly inside cross-validation, and one extra trick people use to stop small groups from being over-trusted.

**Your answer (Q5):**

**Review (Q5):**

**Model answer (you skipped this one).**

* **(a)** The first number is higher because the held-out houses' own prices are inside their neighborhood means: **leakage**. The 0.470 train-only number is the honest estimate for new data.
* **(b)** In smaller groups a house's own price is a bigger share of its group mean (one-third in a 3-home group, all of it in a 1-home group), so the leak grows.
* **(c)** Compute the means inside each CV fold from the training folds only (out-of-fold for training rows), then apply them to the validation fold. Common trick: **smoothing**, blending each group mean with the overall mean, weighted by group size, so small groups aren't over-trusted.